In [7]:
import pandas as pd

# 需要剔除的状态
exclude_status = ['0.Incomplete', '1.In Progress']

# 读取主表
full_df = pd.read_csv(
    "main_analysis_model.csv",
    parse_dates=["sample_datetime"]
)

# 日期筛选
full_df = full_df[full_df["sample_datetime"] >= "2025-09-01"]

# 只读取业务变量表里需要的列，节省内存
status_df = pd.read_csv(
    "business_analysis_variable_library.csv",
    usecols=["application_id", "application_status"]
)

# 如果 application_id 有重复，保留一条
status_df = status_df.drop_duplicates(subset=["application_id"])

# 关联 application_status
full_df = full_df.merge(
    status_df,
    on="application_id",
    how="left"
)

# 过滤 application_status
full_df = full_df[~full_df["application_status"].isin(exclude_status)]

# 保存
full_df.to_csv("main_analysis_model_filtered.csv", index=False)

print(full_df.shape)
print(full_df["application_status"].value_counts(dropna=False))

(22621, 25)
4.Funded    22621
Name: application_status, dtype: int64


In [ ]:
帮我写一个代码，以application_id为主键，只保留main_analysis_model.csv中存在的application_id，所有的文件

In [8]:
import os
import pandas as pd

# INPUT目录
input_dir = "/opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT"

# 主文件
main_file = os.path.join(input_dir, "main_analysis_model.csv")

# 新文件夹（所有过滤后的文件放这里）
output_dir = os.path.join(input_dir, "filtered_files_by_application_id")
os.makedirs(output_dir, exist_ok=True)

# 读取主表 application_id
print("读取 main_analysis_model.csv ...")

main_df = pd.read_csv(
    main_file,
    usecols=["application_id"]
)

valid_app_ids = set(
    main_df["application_id"]
    .dropna()
    .astype(str)
    .unique()
)

print(f"主表 application_id 数量: {len(valid_app_ids)}")

# 遍历 INPUT 下所有 csv
for file_name in os.listdir(input_dir):

    # 只处理 csv
    if not file_name.endswith(".csv"):
        continue

    file_path = os.path.join(input_dir, file_name)

    try:
        print(f"\n处理中: {file_name}")

        df = pd.read_csv(file_path)

        # 没有 application_id 的文件直接跳过
        if "application_id" not in df.columns:
            print(f"跳过（无 application_id）: {file_name}")
            continue

        # 转字符串避免类型不一致
        df["application_id"] = df["application_id"].astype(str)

        # 过滤
        filtered_df = df[
            df["application_id"].isin(valid_app_ids)
        ]

        # 输出到新文件夹
        output_path = os.path.join(output_dir, file_name)

        filtered_df.to_csv(output_path, index=False)

        print(f"原始行数: {len(df)}")
        print(f"保留行数: {len(filtered_df)}")
        print(f"保存完成: {output_path}")

    except Exception as e:
        print(f"处理失败 {file_name}: {e}")

print("\n全部处理完成")

读取 main_analysis_model.csv ...
主表 application_id 数量: 65178

处理中: occupation_information.csv
原始行数: 151574
保留行数: 22620
保存完成: /opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT/filtered_files_by_application_id/occupation_information.csv

处理中: model_variable_library.csv
原始行数: 788513
保留行数: 53555
保存完成: /opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT/filtered_files_by_application_id/model_variable_library.csv

处理中: main_analysis_model.csv
原始行数: 65178
保留行数: 65178
保存完成: /opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT/filtered_files_by_application_id/main_analysis_model.csv

处理中: business_analysis_variable_library.csv


/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:2882: DtypeWarning: Columns (31) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


原始行数: 2401826
保留行数: 65178
保存完成: /opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT/filtered_files_by_application_id/business_analysis_variable_library.csv

处理中: cross_model_score.csv


/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:2882: DtypeWarning: Columns (4,10,21,22) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


原始行数: 860661
保留行数: 64995
保存完成: /opt/workspace/aus_group_drive/Eliam/cdaml_bigq/modeling/model_analysis_project/INPUT/filtered_files_by_application_id/cross_model_score.csv

全部处理完成
